## Imports

In [1]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host= 
    # 'bi-presto.serving.data.production.internal',
    # 'processing-2.processing.data.production.internal',
    'presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [4]:
print(os.getcwd())
local_dataset = '/Users/E2074/local-datasets/customer/geo-consistency'
print(local_dataset)

/Users/E2074/analytics/customer/segmentation/geo-consistency
/Users/E2074/local-datasets/customer/geo-consistency


## Datasets & Parameter

In [23]:
city = 'Bangalore'
date_str = '20250316'
date_obj = datetime.strptime(date_str, "%Y%m%d")

start_date = date_obj - timedelta(days=17)
end_date = date_obj

date_list = [(start_date + timedelta(days=i)).strftime("%Y%m%d") for i in range(18)]

print("start_date =", start_date.strftime("%Y%m%d"))
print("end_date =", end_date.strftime("%Y%m%d"))
print("date_range =", len(date_list))
print("list_date =", date_list)

start_date = 20250227
end_date = 20250316
date_range = 18
list_date = ['20250227', '20250228', '20250301', '20250302', '20250303', '20250304', '20250305', '20250306', '20250307', '20250308', '20250309', '20250310', '20250311', '20250312', '20250313', '20250314', '20250315', '20250316']


In [24]:
for date in date_list:

    query_start_time = time.time()
    base_query = f"""

    with base as (
    select 
        customer_id
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = '{city}'
    ),

    use_case as (
    select 
        hex_12,
        primary_tag
    from 
        experiments_internal.combined_geo_usecase_hex_12_level 
    where 
        current_city = '{city}'
    
    ),
    
    orders as (
    select
        yyyymmdd,
        customer_id,
        service_obj_service_name service_name,
        order_id,
        order_status,
        pickup_cluster,
        drop_cluster,
        pickup_location_hex_8,
        drop_location_hex_8,
        pickup_location_hex_10,
        drop_location_hex_10,
        pickup_location_hex_12,
        drop_location_hex_12,
        channel_host
    from 
        orders.order_logs_snapshot
    where 
        yyyymmdd = '{date}'
        and city_name = '{city}'
        and service_obj_service_name in ('Auto', 'Link', 'CabEconomy', 'Bike Lite', 'CabPremium', 'Cab SUV', 'Auto Pet', 'Auto C2C', 'Auto NCR','Auto Pool')
        and channel_host in ('android','ios')
    )

    select
        orders.*,
        pickup.primary_tag pickup_tag,
        drop_.primary_tag drop_tag
    from 
        base
    join 
        orders
        on base.customer_id = orders.customer_id
    left join 
        use_case pickup
        on pickup.hex_12 = pickup_location_hex_12
    left join 
        use_case drop_
        on drop_.hex_12 = drop_location_hex_12
    
    """
    
    df_base_query = pd.read_sql(base_query, connection)
    query_end_time = time.time()
    
    extract_start_time = time.time()
    df_base_query.to_parquet(local_dataset + '/ban_gc_data_dump_{}.parquet'.format(date), index=False)
    extract_end_time = time.time()
    
    query_execution_time = query_end_time - query_start_time
    local_extract_time = extract_end_time - extract_start_time
    
    print(f"Query execution time: {query_execution_time:.6f} seconds & local extract time: {local_extract_time:.6f} seconds ")

Query execution time: 60.630638 seconds & local extract time: 3.499231 seconds 
Query execution time: 70.963092 seconds & local extract time: 2.748857 seconds 
Query execution time: 75.203154 seconds & local extract time: 2.467795 seconds 
Query execution time: 57.307338 seconds & local extract time: 1.660370 seconds 
Query execution time: 55.411538 seconds & local extract time: 2.008063 seconds 
Query execution time: 56.639362 seconds & local extract time: 1.869999 seconds 
Query execution time: 58.810340 seconds & local extract time: 2.007714 seconds 
Query execution time: 57.917185 seconds & local extract time: 1.877080 seconds 
Query execution time: 70.835315 seconds & local extract time: 2.065157 seconds 
Query execution time: 62.448172 seconds & local extract time: 2.992244 seconds 
Query execution time: 51.952166 seconds & local extract time: 2.287754 seconds 
Query execution time: 59.627298 seconds & local extract time: 2.585981 seconds 
Query execution time: 58.013385 seconds 

In [1]:
import pandas as pd

# Sample DataFrame with multiple customers
data = {
    "customer_id": ["xyz", "xyz", "xyz", "xyz", "abc", "abc", "abc", "pqr", "pqr", "pqr"],
    "order_id": range(1, 11),
    "pickup_location_hex_8": ["A", "A", "B", "B", "A", "B", "C", "D", "E", "F"],
    "drop_location_hex_8": ["B", "B", "A", "A", "B", "A", "D", "E", "F", "G"],
}

df = pd.DataFrame(data)


In [2]:
df

,customer_id,order_id,pickup_location_hex_8,drop_location_hex_8
0,xyz,1,A,B
1,xyz,2,A,B
2,xyz,3,B,A
3,xyz,4,B,A
4,abc,5,A,B
5,abc,6,B,A
6,abc,7,C,D
7,pqr,8,D,E
8,pqr,9,E,F
9,pqr,10,F,G


In [3]:

# Use frozenset to normalize routes (vectorized approach)
df["route"] = [frozenset(x) for x in zip(df["pickup_location_hex_8"], df["drop_location_hex_8"])]
df

,customer_id,order_id,pickup_location_hex_8,drop_location_hex_8,route
0,xyz,1,A,B,"(A, B)"
1,xyz,2,A,B,"(A, B)"
2,xyz,3,B,A,"(A, B)"
3,xyz,4,B,A,"(A, B)"
4,abc,5,A,B,"(A, B)"
5,abc,6,B,A,"(A, B)"
6,abc,7,C,D,"(C, D)"
7,pqr,8,D,E,"(E, D)"
8,pqr,9,E,F,"(E, F)"
9,pqr,10,F,G,"(F, G)"


In [4]:

# Count occurrences of each route per customer
route_counts = df.groupby(["customer_id", "route"]).size().reset_index(name="count")
route_counts


,customer_id,route,count
0,abc,"(A, B)",2
1,abc,"(C, D)",1
2,pqr,"(E, D)",1
3,pqr,"(E, F)",1
4,pqr,"(F, G)",1
5,xyz,"(A, B)",4


In [5]:
# Total orders per customer
total_orders_per_customer = df["customer_id"].value_counts().reset_index()
total_orders_per_customer

,customer_id,count
0,xyz,4
1,abc,3
2,pqr,3


In [6]:
total_orders_per_customer.columns = ["customer_id", "total_orders"]
total_orders_per_customer

,customer_id,total_orders
0,xyz,4
1,abc,3
2,pqr,3


In [7]:
# Merge and compute percentage per customer
route_counts = route_counts.merge(total_orders_per_customer, on="customer_id")
route_counts

,customer_id,route,count,total_orders
0,abc,"(A, B)",2,3
1,abc,"(C, D)",1,3
2,pqr,"(E, D)",1,3
3,pqr,"(E, F)",1,3
4,pqr,"(F, G)",1,3
5,xyz,"(A, B)",4,4


In [8]:
route_counts["percentage"] = route_counts["count"] / route_counts["total_orders"]
route_counts

,customer_id,route,count,total_orders,percentage
0,abc,"(A, B)",2,3,0.666667
1,abc,"(C, D)",1,3,0.333333
2,pqr,"(E, D)",1,3,0.333333
3,pqr,"(E, F)",1,3,0.333333
4,pqr,"(F, G)",1,3,0.333333
5,xyz,"(A, B)",4,4,1.000000
